## Dataset Preparation

In [ ]:
!pip install transformers
!pip install -q transformers datasets
!pip install transformers[torch]

In [ ]:
!pip install sentencepiece
!pip install sacremoses

In [ ]:
import pandas as pd
from google.colab import drive
import torch
import torch.nn as nn
import numpy as np
import ast
from transformers import MarianMTModel, MarianTokenizer

In [ ]:
if torch.cuda.is_available():
    device = torch.cuda.get_device_name(0)
    print('GPU device:', device)
else:
    print('No GPU available.')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU device: Tesla T4


In [ ]:
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
train_subset01 = pd.read_csv("/content/gdrive/My Drive/MSc Project/train_subset.csv", converters={'labels': ast.literal_eval})
train_subset05 = pd.read_csv("/content/gdrive/My Drive/MSc Project/train_subset5.csv", converters={'labels': ast.literal_eval})
train_subset10 = pd.read_csv("/content/gdrive/My Drive/MSc Project/train_subset10.csv", converters={'labels': ast.literal_eval})
train_subset20 = pd.read_csv("/content/gdrive/My Drive/MSc Project/train_subset20.csv", converters={'labels': ast.literal_eval})
train_subset30 = pd.read_csv("/content/gdrive/My Drive/MSc Project/train_subset30.csv", converters={'labels': ast.literal_eval})

## Backtranslation : Spanish


In [ ]:
# Initialize the MarianMT models for English <-> Spanish languages
target_model_name = 'Helsinki-NLP/opus-mt-en-es'
target_tokenizer = MarianTokenizer.from_pretrained(target_model_name)
target_model = MarianMTModel.from_pretrained(target_model_name).to('cuda')

en_model_name = 'Helsinki-NLP/opus-mt-es-en'
en_tokenizer = MarianTokenizer.from_pretrained(en_model_name)
en_model = MarianMTModel.from_pretrained(en_model_name).to('cuda')

In [ ]:
# Function to perform translation using the MarianMT models
def translate(texts, model, tokenizer, language="es"):
    template = lambda text: f"{text}" if language == "en" else f">>{language}<< {text}"
    src_texts = [template(text) for text in texts]
    encoded = tokenizer.prepare_seq2seq_batch(src_texts, return_tensors='pt').to('cuda')
    translated = model.generate(**encoded)
    translated_texts = tokenizer.batch_decode(translated, skip_special_tokens=True)
    return translated_texts

# Function to perform back-translation using the MarianMT models
def back_translate(texts, target_lang="es", source_lang="en"):
    es_texts = translate(texts, target_model, target_tokenizer, language=target_lang)
    back_translated_texts = translate(es_texts, en_model, en_tokenizer, language=source_lang)
    return back_translated_texts

In [ ]:
def backtranslated_df(train_df, augmentation_percentage):
    num_rows = train_df.shape[0]
    num_augment = int(num_rows * (augmentation_percentage / 100))

    # Sample the rows to augment
    sample_df = train_df.sample(n=num_augment, random_state=42).copy() # deep copy

    for idx in sample_df.index:
        text_to_augment = sample_df.loc[idx, 'text'] # use sample_df instead

        # Back translate the text
        augmented_text = back_translate([text_to_augment])[0]

        # Replace the text in the sample_df
        sample_df.loc[idx, 'text'] = augmented_text

    # Append the augmented data to the original dataframe
    train_df = pd.concat([train_df, sample_df], ignore_index=True)

    return train_df


In [ ]:
# Apply 20% augmentation on 'text' column and save to CSV
subset_backtrans_20 = backtranslated_df(train_subset, 20)
subset_backtrans_20.to_csv('/content/gdrive/My Drive/MSc Project/subset_backtrans_20.csv', index=False)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:3761: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and targets.

Here is a short example:

model_inputs = tokenizer(src_texts, text_target=tgt_texts, ...)

If you either need to use different keyword arguments for the source and target texts, you should do two calls like
this:

model_inputs = tokenizer(src_texts, ...)
labels = tokenizer(text_target=tgt_texts, ...)
model_inputs["labels"] = labels["input_ids"]

See the documentation of your specific tokenizer for more details on the specific arguments to the tokenizer of choice.
For a more complete example, see the implementation of `prepare_seq2seq_batch`.

  warnings.warn(formatted_warning, FutureWarning)
/usr/local/lib/python3.10/dist-packages/transformers/generation/utils.py:1369: UserWarning: Using `max_length`'s default (512) t

In [ ]:
# Apply 40% augmentation on 'text' column and save to CSV
subset_backtrans_40 = backtranslated_df(train_subset, 40)
subset_backtrans_40.to_csv('/content/gdrive/My Drive/MSc Project/subset_backtrans_40.csv', index=False)

In [ ]:
# Apply 60% augmentation on 'text' column and save to CSV
subset5_backtrans_60 = backtranslated_df(train_subset, 60)
subset5_backtrans_60.to_csv('/content/gdrive/My Drive/MSc Project/subset5_backtrans_60.csv', index=False)

In [ ]:
# Apply 80% augmentation on 'text' column and save to CSV
subset_backtrans_80 = backtranslated_df(train_subset, 80)
subset_backtrans_80.to_csv('/content/gdrive/My Drive/MSc Project/subset_backtrans_80.csv', index=False)

In [ ]:
# Apply 100% augmentation on 'text' column and save to CSV
subset_backtrans_100 = backtranslated_df(train_subset, 100)
subset_backtrans_100.to_csv('/content/gdrive/My Drive/MSc Project/subset_backtrans_100.csv', index=False)

In [ ]:
# Apply 60% augmentation on 'text' column and save to CSV
subset10_backtrans_60 = backtranslated_df(train_subset10, 60)
subset10_backtrans_60.to_csv('/content/gdrive/My Drive/MSc Project/subset10_backtrans_60.csv', index=False)

In [ ]:
# Apply 60% augmentation on 'text' column and save to CSV
subset20_backtrans_60 = backtranslated_df(train_subset20, 60)
subset20_backtrans_60.to_csv('/content/gdrive/My Drive/MSc Project/subset20_backtrans_60.csv', index=False)

In [ ]:
# Apply 60% augmentation on 'text' column and save to CSV
subset30_backtrans_60 = backtranslated_df(train_subset30, 60)
subset30_backtrans_60.to_csv('/content/gdrive/My Drive/MSc Project/subset30_backtrans_60.csv', index=False)

## Backtranslation: Korean

In [ ]:
pip install --upgrade google-cloud-translate


In [ ]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/content/gdrive/My Drive/fluted-reason-395619-a9a9f7c75f9f.json"
from google.cloud import translate_v2 as translate

In [ ]:
# Initialize the client
client = translate.Client()

def back_translate(texts, intermediate_lang='ko'):
    # Translate to the intermediate language
    intermediate_texts = [client.translate(text, target_language=intermediate_lang)['translatedText'] for text in texts]

    # Translate back to the original language
    back_translated_texts = [client.translate(text, source_language=intermediate_lang)['translatedText'] for text in intermediate_texts]

    return back_translated_texts

def backtranslated_df(train_df, augmentation_percentage):
    num_rows = train_df.shape[0]
    num_augment = int(num_rows * (augmentation_percentage / 100))

    # Sample the rows to augment
    sample_df = train_df.sample(n=num_augment, random_state=42).copy()  # deep copy

    for idx in sample_df.index:
        text_to_augment = sample_df.loc[idx, 'text']

        # Back translate the text
        augmented_text = back_translate([text_to_augment])[0]

        # Replace the text in the sample_df
        sample_df.loc[idx, 'text'] = augmented_text

    # Append the augmented data to the original dataframe
    train_df = pd.concat([train_df, sample_df], ignore_index=True)

    return train_df


In [ ]:
# Apply 20% augmentation on 'text' column and save to CSV
subset_backtrans_ko20 = backtranslated_df(train_subset, 20)
subset_backtrans_ko20.to_csv('/content/gdrive/My Drive/MSc Project/subset_backtrans_ko20.csv', index=False)

In [ ]:
# Apply 40% augmentation on 'text' column and save to CSV
subset_backtrans_ko40 = backtranslated_df(train_subset, 40)
subset_backtrans_ko40.to_csv('/content/gdrive/My Drive/MSc Project/subset_backtrans_ko40.csv', index=False)

In [ ]:
# Apply 60% augmentation on 'text' column and save to CSV
subset_backtrans_ko60 = backtranslated_df(train_subset, 60)
subset_backtrans_ko60.to_csv('/content/gdrive/My Drive/MSc Project/subset_backtrans_ko60.csv', index=False)

In [ ]:
# Apply 80% augmentation on 'text' column and save to CSV
subset_backtrans_ko80 = backtranslated_df(train_subset, 80)
subset_backtrans_ko80.to_csv('/content/gdrive/My Drive/MSc Project/subset_backtrans_ko80.csv', index=False)

In [ ]:
# Apply 100% augmentation on 'text' column and save to CSV
subset_backtrans_ko100 = backtranslated_df(train_subset, 100)
subset_backtrans_ko100.to_csv('/content/gdrive/My Drive/MSc Project/subset_backtrans_ko100.csv', index=False)

In [ ]:
# Apply 60% augmentation on 'text' column and save to CSV
subset05_backtrans_ko60 = backtranslated_df(train_subset05, 60)
subset05_backtrans_ko60.to_csv('/content/gdrive/My Drive/MSc Project/subset05_backtrans_ko60.csv', index=False)

In [ ]:
# Apply 60% augmentation on 'text' column and save to CSV
subset10_backtrans_ko60 = backtranslated_df(train_subset10, 60)
subset10_backtrans_ko60.to_csv('/content/gdrive/My Drive/MSc Project/subset10_backtrans_ko60.csv', index=False)

In [ ]:
# Apply 60% augmentation on 'text' column and save to CSV
subset20_backtrans_ko60 = backtranslated_df(train_subset20, 60)
subset20_backtrans_ko60.to_csv('/content/gdrive/My Drive/MSc Project/subset20_backtrans_ko60.csv', index=False)

In [ ]:
# Apply 60% augmentation on 'text' column and save to CSV
subset30_backtrans_ko60 = backtranslated_df(train_subset30, 60)
subset30_backtrans_ko60.to_csv('/content/gdrive/My Drive/MSc Project/subset30_backtrans_ko60.csv', index=False)